### Executive Summary

在 Heston (1993) 隨機波動率模型中，常態市場校準結果往往顯示 Feller 條件不滿足 ($2\kappa\theta < \xi^2$，即 Feller Ratio $< 1$)。這是因為真實市場選擇權的隱含波動率微笑 (volatility smile) 具有極高的曲率，需要較大的波動率之波動率 (vol-of-vol, $\xi$) 才能擬合。

在定價標準歐式選擇權時，Feller 條件失效的影響微乎其微（因為解析特徵函數不依賴 Feller，且 Monte Carlo 的終端股價分布會均化個別路徑的微小偏差）。然而，對於變異數交換 (Variance Swap) 等路徑依賴型衍生品，離散化格式偏誤 (Discretization Bias) 會直接轉化為定價與 P&L 的顯著誤差。

本分析主要針對以下核心重點進行探討：

**1. 歐式選擇權 vs. 變異數交換的結構差異**
- 歐式選擇權：Payoff 僅取決於終端價格 $S_T$。路徑過程中的微小偏誤在大量路徑累加後會被平均抵消。
- 變異數交換：Payoff 取決於整條路徑的已實現變異數 $\int_0^T v_t \, dt$（或離散版本的 $\sum (\Delta \ln S)^2$）。路徑偏誤不會被抵消，而是直接線性累加進合約收益中。

**2. Full Truncation 的陷阱**
當 Feller Ratio $< 1$ 時，變異數過程 $v_t$ 理論上會頻繁觸碰或貼近 0 邊界。傳統 Euler-Maruyama 加上 Full Truncation $\max(v_t, 0)$ 或反射機制在粗時間步長下，會產生嚴重的分佈失真與期望值的系統性偏誤。

**3. Andersen (2008) Quadratic-Exponential (QE) 方案**
通過精確匹配非中心卡方分佈的前兩階矩，並在低變異數區域採用混合指數分佈（包含 0 點機率質量），該方案能在任意粗細的時間步長下有效消除邊界偏誤。

**4. 雙重偏誤解耦 (Decoupling Biases)**
- 模擬格式偏誤 (Discretization Scheme Bias)：Full Truncation vs. QE scheme（與 Feller 條件密切相關）。
- 監控頻率偏誤 (Monitoring Frequency Bias)：連續積分 $\frac{1}{T}\int_0^T v_t dt$ vs. 離散每日抽樣 $\frac{252}{n_{obs}}\sum (\Delta \ln S)^2$（與合約規格及 Jensen 不等式相關）。

---

### Closed Form Variance Swap

根據 CIR 過程的漂移項微分方程：
$$\frac{d}{dt} \mathbb{E}[v_t] = \kappa (\theta - \mathbb{E}[v_t]) \implies \mathbb{E}[v_t] = \theta + (v_0 - \theta) e^{-\kappa t}$$

對其在 $[0, T]$ 積分並除以 $T$，可得連續公平執行價的封閉解析解：
$$K_{\text{var}}^{\text{cont}} = \mathbb{E}\left[\frac{1}{T} \int_0^T v_t \, dt\right] = \theta + (v_0 - \theta) \frac{1 - e^{-\kappa T}}{\kappa T}$$

獨立於 $\xi$ 與 $\rho$：連續公平執行價僅取決於 $\kappa, \theta, v_0$ 與天期 $T$。即使 vol-of-vol $\xi$ 極大或 Feller 條件嚴重失效，連續解析值 $K_{\text{var}}^{\text{cont}}$ 始終是準確的。

---

### Three-Way Realized Variance Comparison

在日常交易實務中，變異數交換通常依據每日收盤價計算已實現變異數。我們在標準天期 $T = 1.0$ 年、日常監控步長 ($N = 252$ 步)、$50,000$ 條模擬路徑下，比較三種定價方法：

1. 解析封閉解 (Analytical Benchmark)：客觀真實基準。
2. MC + Full Truncation (FT)：傳統 Euler 截斷近似。
3. MC + Andersen QE Scheme：自適應矩匹配近似。

評估指標：
- 連續變異數 (Continuous RV)：$\frac{1}{T} \int_0^T v_t dt$
- 離散變異數 (Discrete RV)：$\frac{1}{T} \sum_{i=1}^{n_{obs}} (\Delta \ln S_i)^2$
- 定價誤差 (Basis Points, bps)：$(\widehat{K}_{\text{var}} - K_{\text{var}}^{\text{exact}}) \times 10^4$

---

### Convergence Analysis Across Time Steps

Feller 條件失效時，離散時間步長越粗（$\Delta t$ 越大），Euler-Maruyama Full Truncation 的邊界偏誤越劇烈。

在本實驗中，我們固定模擬路徑數在多組時間步長下測試收斂性：
$$N_{\text{steps}} \in [12, 52, 100, 252, 500]$$

我們檢驗：
1. Full Truncation 在步數少時是否出現幾十個 bps 的巨大偏誤。
2. Andersen QE Scheme 是否在任意步長下均能精確貼合解析解。

---

### Decoupling the Biases: Discretization Scheme vs. Monitoring Frequency

| 偏誤類型 | 數學來源 | 與模擬格式關係 | 與 Feller 條件關係 | 實務解決方案 |
| :--- | :--- | :--- | :--- | :--- |
| **模擬格式偏誤<br>(Discretization Scheme Bias)** | Euler-Maruyama 截斷近似無法處理 $v_t \to 0$ 的非中心卡方極限 | 極大：FT 偏誤顯著，QE 幾乎無偏 | 極大：Feller 比率越小，偏誤越嚴重 | 更換模擬演算法<br>(改用 Andersen QE Scheme) |
| **監控頻率偏誤<br>(Monitoring Frequency Bias)** | 離散回報平方和 $\sum (\Delta \ln S)^2$ 與連續積分 $\int v_t dt$ 的差距 (Itô-Doeblin 修正項) | 無關：即便使用精確解 (Broadie-Kaya) 也存在此客觀差距 | 次要：主要受標的價格凸性與無風險利率漂移影響 | 加入合約離散凸性調整項<br>(Discrete Monitoring Adjustment) |

**離散回報平方和的理論修正**

由 Itô 引理：
$$\ln \frac{S_{t+\Delta t}}{S_t} = \left(r - \frac{1}{2} v_t\right)\Delta t + \sqrt{v_t} \Delta W_t^S$$

對其取平方的期望值：
$$\mathbb{E}\left[ \left(\ln \frac{S_{t+\Delta t}}{S_t}\right)^2 \right] = \mathbb{E}[v_t]\Delta t + \mathbb{E}\left[\left(r - \frac{1}{2}v_t\right)^2\right] \Delta t^2$$

因此，離散已實現變異數 $\frac{1}{T}\sum (\Delta \ln S)^2$ 相較於連續積分 $\frac{1}{T}\int v_t dt$，本身就帶有 $\mathcal{O}(\Delta t)$ 等級的監控頻率偏差。

### Feller Ratio 與時間步長收斂性

這兩張圖表與收斂數據，主要展示當 Feller Ratio < 1（Feller 條件嚴重失效）時，不同數值模擬方法的穩定度與計算效率差異。這提供了三個重要的量化定價洞見：

1. Full Truncation 對時間步長極度敏感
當變異數頻繁逼近 0 邊界時，Full Truncation 在粗糙網格下（例如 12 步或 52 步）直接截斷負值，會產生巨大的系統性期望值誤差。為了讓誤差收斂到可接受的水準，被迫必須將時間步長切得非常細（例如 252 步或 500 步）。

2. Andersen QE 方案展現絕佳的魯棒性
Andersen QE 演算法精確匹配了非中心卡方分佈的矩，並妥善處理了零點機率質量。因此在任何時間步長下，即便只有極端粗糙的 12 步，定價誤差都能維持在一個相對穩定且較低的水準，從根本上消除了逼近 0 邊界時產生的離散化偏誤。

3. 實務上的計算效率解放
在 Monte Carlo 模擬中，計算成本與時間步數呈線性正比。在實務常態違反 Feller 條件的環境中，若使用 Full Truncation，必須耗費大量計算資源跑極細的步長來確保精度；若改用 Andersen QE Scheme，只需極少的步長就能得到可靠的定價結果。這意味著在需要即時報價或計算大批風險參數時，Andersen QE 能大幅度節省計算時間與資源。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from heston import (
    HestonParameters,
    AnalyticalVarianceSwapPricer,
    HestonSimulatorFT,
    HestonSimulatorQE
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'figure.figsize': (12, 5.5),
    'figure.dpi': 140,
    'font.size': 11,
    'lines.linewidth': 2.0,
    'grid.alpha': 0.35,
    'grid.linestyle': '--'
})

pd.set_option('display.float_format', lambda x: f'{x:.6f}')

: 

In [ ]:
v_0 = 0.09
base_params = HestonParameters(
    kappa=1.70,
    theta=0.04,
    xi=0.60,
    rho=-0.70,
    v_0=v_0
)

print(f"Feller Ratio: {base_params.feller_ratio:.4f}")
print(f"Feller Condition Satisfied: {base_params.is_feller_satisfied}")
display(base_params.summary())

In [ ]:
# Analytical Continuous Variance Strike Pricing
S0 = 120.0
T = 1.0
r = 0.02
q = 0
n_steps = 252
n_sims = 100_000
seed = 12345

market_inputs_sim = {
    "S0": S0,
    "T": T,
    "r": r,
    "n_steps":n_steps,
    "n_sims":n_sims
}

exact_kvar = AnalyticalVarianceSwapPricer.fair_variance_strike(base_params, T)
exact_vol_strike = AnalyticalVarianceSwapPricer.fair_volatility_strike(base_params, T)

print(f"--- Continuous Variance Swap Analytical Benchmark (T = {T} yr) ---")
print(f"Exact Fair Variance Strike (K_var) : {exact_kvar:.6f}")
print(f"Exact Fair Volatility Strike       : {exact_vol_strike * 100:.3f}%")

In [ ]:
# Compare Continuous Integral vs Discrete Daily Sum by FT and by Andersen QE Scheme under simulation

print(f"Running simulation with {n_sims:,} paths and {n_steps} daily monitoring steps...")

sim_ft = HestonSimulatorFT(base_params, seed=seed)
ft_cont, ft_disc = sim_ft.simulate_realized_variance(S0, T, r, n_steps=n_steps, n_sims=n_sims)

sim_qe = HestonSimulatorQE(base_params, seed=seed)
qe_cont, qe_disc = sim_qe.simulate_realized_variance(S0, T, r, n_steps=n_steps, n_sims=n_sims)

mean_ft_cont, mean_ft_disc = np.mean(ft_cont), np.mean(ft_disc)
se_ft_cont, se_ft_disc = np.std(ft_cont) / np.sqrt(n_sims), np.std(ft_disc) / np.sqrt(n_sims)

mean_qe_cont, mean_qe_disc = np.mean(qe_cont), np.mean(qe_disc)
se_qe_cont, se_qe_disc = np.std(qe_cont) / np.sqrt(n_sims), np.std(qe_disc) / np.sqrt(n_sims)

df_experiment_1 = pd.DataFrame({
    'Pricing Method': [
        'Benchmark',
        'FT (Continuous Integral)',
        'FT (Discrete Daily Sum)',
        'QE (Continuous Integral)',
        'QE (Discrete Daily Sum)'
    ],
    'Realized Variance (K_var)': [
        exact_kvar,
        mean_ft_cont,
        mean_ft_disc,
        mean_qe_cont,
        mean_qe_disc
    ],
    'Standard Error (SE)': [
        0.0,
        se_ft_cont,
        se_ft_disc,
        se_qe_cont,
        se_qe_disc
    ],
    'Realized Volatility': [
        np.sqrt(exact_kvar) * 100,
        np.sqrt(mean_ft_cont) * 100,
        np.sqrt(mean_ft_disc) * 100,
        np.sqrt(mean_qe_cont) * 100,
        np.sqrt(mean_qe_disc) * 100
    ],
    'Bias vs. Benchmark (bps)': [
        0.0,
        (mean_ft_cont - exact_kvar) * 10000,
        (mean_ft_disc - exact_kvar) * 10000,
        (mean_qe_cont - exact_kvar) * 10000,
        (mean_qe_disc - exact_kvar) * 10000
    ],
    'Relative Error (%)': [
        0.0,
        (mean_ft_cont - exact_kvar) / exact_kvar * 100,
        (mean_ft_disc - exact_kvar) / exact_kvar * 100,
        (mean_qe_cont - exact_kvar) / exact_kvar * 100,
        (mean_qe_disc - exact_kvar) / exact_kvar * 100
    ]
})

display(df_experiment_1)


In [ ]:
# Compare Step-Size Convergence & Discretization Bias

steps_grid = [12, 52, 100, 252, 500]
convergence_records = []

print(f"Using {n_sims:,} trials to evaluate step size convergence for Full Truncation (FT) and Andersen QE (QE)...")

for steps in steps_grid:
    sim_ft_step = HestonSimulatorFT(base_params, seed=steps + 100)
    ft_c, _ = sim_ft_step.simulate_realized_variance(S0, T, r, steps, n_sims=n_sims)
    
    sim_qe_step = HestonSimulatorQE(base_params, seed=steps + 100)
    qe_c, _ = sim_qe_step.simulate_realized_variance(S0, T, r, steps, n_sims=n_sims)
    
    m_ft_c = np.mean(ft_c)
    se_ft_c = np.std(ft_c) / np.sqrt(n_sims)
    m_qe_c = np.mean(qe_c)
    se_qe_c = np.std(qe_c) / np.sqrt(n_sims)
    
    convergence_records.append({
        "Steps": steps,
        "Years": T / steps,
        "FT_Kvar": f"{m_ft_c:.6f} (SE: {se_ft_c:.6f})",
        "FT_Kvar": m_ft_c,
        "FT_Abs_Error_bps": np.abs(m_ft_c - exact_kvar) * 10000,
        "QE_Kvar": f"{m_qe_c:.6f} (SE: {se_qe_c:.6f})",
        "QE_Kvar": m_qe_c,
        "QE_Abs_Error_bps": np.abs(m_qe_c - exact_kvar) * 10000
    })

df_conv = pd.DataFrame(convergence_records)
display(df_conv)

# --- Dual-Panel High-Quality Visualization ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.2))

# Plot 1: Estimated Variance vs Step Count
ax1.axhline(exact_kvar, color='crimson', linestyle='--', label=f'Analytical Benchmark ({exact_kvar:.5f})', zorder=4)
ax1.plot(df_conv['Steps'], df_conv['FT_Kvar'], marker='o', color='steelblue', label='MC: Full Truncation')
ax1.plot(df_conv['Steps'], df_conv['QE_Kvar'], marker='s', color='forestgreen', label='MC: Andersen QE Scheme')
ax1.set_title('Estimated Fair Variance Strike vs. Time Steps', fontweight='bold', fontsize=12)
ax1.set_xlabel('Number of Time Steps ($N_{steps}$)', fontsize=11)
ax1.set_ylabel('Realized Variance $\\widehat{K}_{var}$', fontsize=11)
ax1.set_xscale('log')
ax1.legend(frameon=True, facecolor='white', framealpha=0.9)
ax1.grid(True, linestyle=':', alpha=0.6)

# Plot 2: Absolute Error (bps) Convergence
ax2.plot(df_conv['Steps'], df_conv['FT_Abs_Error_bps'], marker='o', color='steelblue', label='Full Truncation Error (bps)')
ax2.plot(df_conv['Steps'], df_conv['QE_Abs_Error_bps'], marker='s', color='forestgreen', label='Andersen QE Error (bps)')
ax2.set_title('Absolute Discretization Bias vs. Analytical Benchmark', fontweight='bold', fontsize=12)
ax2.set_xlabel('Number of Time Steps ($N_{steps}$)', fontsize=11)
ax2.set_ylabel('Absolute Error (Basis Points)', fontsize=11)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.legend(frameon=True, facecolor='white', framealpha=0.9)
ax2.grid(True, which='both', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Error Decomposition: Scheme Bias vs. Monitoring Frequency Bias

scheme_bias_ft = (mean_ft_cont - exact_kvar) * 10000
scheme_bias_qe = (mean_qe_cont - exact_kvar) * 10000

monitoring_bias_ft = (mean_ft_disc - mean_ft_cont) * 10000
monitoring_bias_qe = (mean_qe_disc - mean_qe_cont) * 10000

total_bias_ft = (mean_ft_disc - exact_kvar) * 10000
total_bias_qe = (mean_qe_disc - exact_kvar) * 10000

df_bias_decomp = pd.DataFrame({
    "Component": [
        "Continuous MC - Analytical",
        "Discrete Sum - Continuous MC",
        "Total Net Bias vs. Continuous Benchmark"
    ],
    "FT (bps)": [scheme_bias_ft, monitoring_bias_ft, total_bias_ft],
    "QE (bps)": [scheme_bias_qe, monitoring_bias_qe, total_bias_qe],
    "Root Cause": [
        "Euler truncation failure (Feller < 1)",
        "Sampling (Itô squared drift term)",
        "Combined effect"
    ]
})

display(df_bias_decomp)

In [ ]:
# Feller Ratio Bias Chart

xi_vals = np.linspace(0.2, 1.5, 15)
feller_ratios = []
ft_errors = []
qe_errors = []

for xi_val in xi_vals:
    params = HestonParameters(
        kappa=1.70,
        theta=0.04,
        xi=xi_val, # vol of vol
        rho=-0.70,
        v_0=0.09
    )
    feller_ratios.append(params.feller_ratio)
    exact_k = AnalyticalVarianceSwapPricer.fair_variance_strike(params, T)

    sim_ft = HestonSimulatorFT(params, seed)
    ft_cont, _ = sim_ft.simulate_realized_variance(S0, T, r, n_steps=n_steps, n_sims=n_sims)
    mean_ft = np.mean(ft_cont)
    ft_errors.append(abs(mean_ft - exact_k) / exact_k * 100)

    sim_qe = HestonSimulatorQE(params, seed)
    qe_cont, _ = sim_qe.simulate_realized_variance(S0, T, r, n_steps=n_steps, n_sims=n_sims)
    mean_qe = np.mean(qe_cont)
    qe_errors.append(abs(mean_qe - exact_k) / exact_k * 100)

fig, ax = plt.subplots()
ax.plot(feller_ratios, ft_errors, marker='o', label='Euler Full Truncation')
ax.plot(feller_ratios, qe_errors, marker='s', label='Andersen QE')
ax.set_xlabel('Feller Ratio (2*kappa*theta / xi^2)')
ax.set_ylabel('Absolute % Error')
ax.set_title(f'Absolute % Error vs. Feller Ratio (N={n_steps})')
ax.invert_xaxis()
ax.legend()
ax.grid(True)
os.makedirs('assets/img', exist_ok=True)
plt.savefig('assets/img/feller_bias_chart.png', bbox_inches='tight')
plt.show()